In [1]:
%load_ext autoreload
%autoreload 2

import torch
from ncpu.runner  import setup_scheduled_trainer, setup_dynamic_trainer
from ncpu.dataset import NCPUDataset

from matplotlib import pyplot as plt
from IPython.display import clear_output, display
from tqdm.auto import tqdm

model_path = None

In [ ]:
from ncpu.dataset import DynamicDataset
from ncpu.config import TINY_AND_TRAINING_CONFIG

BATCH_SIZE = 8
STEPS = 2_500
trainer = setup_dynamic_trainer(
    TINY_AND_TRAINING_CONFIG, 
    update_y = 2,
    update_x = 0,
    steps = STEPS*BATCH_SIZE,
    stages = 4,
)


# Learn GATE Logic

In [ ]:
stop_loss = 0.0001
pbar = tqdm(range(STEPS))
for i in pbar:
    info = trainer.optim_step(steps=(30, 80))
    loss = info["metrics"]["loss"]
    pbar.set_description(f"loss={loss:.6f}")

    if i % 100 == 0:
        clear_output(wait=False)
        display(pbar.container)

        trainer.display_optim_step(info)
        trainer.save_checkpoint()

    if loss == stop_loss:
        break

trainer.save_checkpoint("AND_positional.pth", steps=False, timestamp=False)

# Learning Information Propagation

In [9]:
# load seed model
trainer.load_checkpoint("ncpu_AND_positional.pth")

In [ ]:
stop_loss = 0.0001
N_evolutions = 5
STEPS = 2000
if trainer.ds.counter < STEPS * BATCH_SIZE:
    trainer.ds.counter = STEPS * BATCH_SIZE
trainer.ds.steps = STEPS * BATCH_SIZE
pbar = tqdm(range(STEPS * N_evolutions))
for i in pbar:
    info = trainer.optim_step(steps=(30, 80))
    loss = info["metrics"]["loss"]
    pbar.set_description(f"loss={loss:.6f}")

    if i % 100 == 0:
        clear_output(wait=False)
        display(pbar.container)

        trainer.display_optim_step(info)
        trainer.save_checkpoint()

    if loss == stop_loss:
        break

In [ ]:

trainer.save_checkpoint("AND_propagated_5.pth", steps=False, timestamp=False)

In [9]:
# load seed model
trainer.load_checkpoint("ncpu_AND_propagated_5.pth")

In [ ]:
stop_loss = 0.0001
STEPS = 2000
pbar = tqdm(range(STEPS*10))
for i in pbar:
    info = trainer.optim_step(steps=(30, 80))
    loss = info["metrics"]["loss"]
    pbar.set_description(f"loss={loss:.6f}")

    if i % 100 == 0:
        clear_output(wait=False)
        display(pbar.container)

        trainer.display_optim_step(info)
        trainer.save_checkpoint()

    if loss == stop_loss:
        break

trainer.save_checkpoint("AND_checkpoint_final.pth", steps=False, timestamp=False)

In [ ]:
stop_loss = 0.0001
pbar = tqdm(range(1))
for i in pbar:
    info = trainer.optim_step(steps=(30, 80))
    loss = info["metrics"]["loss"]
    pbar.set_description(f"loss={loss:.6f}")

    clear_output(wait=False)
    display(pbar.container)

    trainer.display_optim_step(info)
